In [131]:
# IMPORTING LIBRARIES

import pandas as pd
import numpy as np

In [132]:
# LOAD DATASET

customers = pd.read_csv("/Users/alon/Downloads/customers-4.csv")
revenue = pd.read_csv("/Users/alon/Downloads/revenue.csv")
subscriptions = pd.read_csv("/Users/alon/Downloads/subscriptions.csv")

In [133]:
# INSPECTING DATASET STRUCTURE

print("Customer:",customers.shape[0],"Rows &",customers.shape[1],"Columns")
print("Revenue:",revenue.shape[0],"Rows &",revenue.shape[1],"Columns")
print("Subscriptions:",subscriptions.shape[0],"Rows &",subscriptions.shape[1],"Columns")

Customer: 1000 Rows & 6 Columns
Revenue: 988 Rows & 6 Columns
Subscriptions: 988 Rows & 4 Columns


In [134]:
print("----- CUSTOMER -----")
print()
print(customers.info())
print()
print("----- REVENUE -----")
print()
print(revenue.info())
print()
print("----- SUBSCRIPTIONS -----")
print()
print(subscriptions.info())

----- CUSTOMER -----

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   customer_id       1000 non-null   int64 
 1   signup_date       1000 non-null   object
 2   plan_type         1000 non-null   object
 3   monthly_fee       1000 non-null   int64 
 4   acquisition_cost  1000 non-null   int64 
 5   churn_date        168 non-null    object
dtypes: int64(3), object(3)
memory usage: 47.0+ KB
None

----- REVENUE -----

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 988 entries, 0 to 987
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   subscription_id  988 non-null    object
 1   customer_id      988 non-null    int64 
 2   month            988 non-null    object
 3   monthly_fee      988 non-null    int64 
 4   revenue_type     988 non-null    object
 5   amo

In [135]:
print("----- CUSTOMER -----")
print()
print(customers.head())
print()
print("----- REVENUE -----")
print()
print(revenue.head())
print()
print("----- SUBSCRIPTIONS -----")
print()
print(subscriptions.head())

----- CUSTOMER -----

   customer_id signup_date plan_type  monthly_fee  acquisition_cost churn_date
0         1001  2024-11-07     Basic           50                30        NaN
1         1002  2024-06-06     Basic           50                30        NaN
2         1003  2024-12-31     Basic           50                30        NaN
3         1004  2024-11-21       Pro          200               100        NaN
4         1005  2024-08-16       Pro          200               100        NaN

----- REVENUE -----

  subscription_id  customer_id    month  monthly_fee revenue_type  amount
0   S-1020-202410         1020  2024-10          200          MRR     200
1   S-1020-202411         1020  2024-11          200          MRR     200
2   S-1020-202412         1020  2024-12          200          MRR     200
3   S-1020-202501         1020  2025-01          200          MRR     200
4   S-1020-202502         1020  2025-02          200          MRR     200

----- SUBSCRIPTIONS -----

  subscrip

In [136]:
# CONVERTING DATE COLUMNS TO DATETIME

customers['signup_date'] = pd.to_datetime(customers['signup_date'])
customers['churn_date'] = pd.to_datetime(customers['churn_date'])
revenue['month'] = pd.to_datetime(revenue['month'], format = "%Y-%m")
subscriptions['month'] = pd.to_datetime(subscriptions['month'], format = '%Y-%m')

In [137]:
print("----- CUSTOMER -----")
print()
print(customers.info())
print()
print("----- REVENUE -----")
print()
print(revenue.info())
print()
print("----- SUBSCRIPTIONS -----")
print()
print(subscriptions.info())

----- CUSTOMER -----

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   customer_id       1000 non-null   int64         
 1   signup_date       1000 non-null   datetime64[ns]
 2   plan_type         1000 non-null   object        
 3   monthly_fee       1000 non-null   int64         
 4   acquisition_cost  1000 non-null   int64         
 5   churn_date        168 non-null    datetime64[ns]
dtypes: datetime64[ns](2), int64(3), object(1)
memory usage: 47.0+ KB
None

----- REVENUE -----

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 988 entries, 0 to 987
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   subscription_id  988 non-null    object        
 1   customer_id      988 non-null    int64         
 2   month            988 no

In [138]:
# PREPARING "dim_customers"

# Active Customers
customers['is_active'] = customers['churn_date'].isnull().astype(int)

In [139]:
# 2. Calculate tenure_days using datetime objects
cutoff_date = pd.to_datetime('2025-06-30')
customers['end_date'] = customers['churn_date'].fillna(cutoff_date)
customers['tenure_days'] = (customers['end_date'] - customers['signup_date']).dt.days

# Active customer tenure calculation
active_tenure = (
    (customers['end_date'].dt.year - customers['signup_date'].dt.year) * 12
    + (customers['end_date'].dt.month - customers['signup_date'].dt.month)
    + 1
)

# Calculate "tenure_month" 
customers['tenure_month'] = ((customers['end_date'].dt.year - customers['signup_date'].dt.year) * 12
                            + 
                            (customers['end_date'].dt.month - customers['signup_date'].dt.month)
                            + 1
                            ).astype(int)

# Converting date columns to date-only
customers['signup_date'] =  customers['signup_date'].dt.date
customers['churn_date'] = customers['churn_date'].dt.date

In [140]:
# Group customers into tenure ranges 
bins = [-1, 3, 6, 9, 12, 18]
labels = ['0-3 Months', '3-6 Months', '6-9 Months', '9-12 Months', '12-18 Months']

# Assigning each customer to a tenure cohort
customers['tenure_cohort'] = pd.cut(customers['tenure_month'], bins=bins, labels=labels)

In [141]:
# Calculating cumulative subscription revenue
customers['total_subscription_revenue'] = customers['tenure_month'] * customers['monthly_fee']

# Clean up temporary columns
customers.drop(columns=['end_date'], inplace=True)

In [142]:
customers.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 11 columns):
 #   Column                      Non-Null Count  Dtype   
---  ------                      --------------  -----   
 0   customer_id                 1000 non-null   int64   
 1   signup_date                 1000 non-null   object  
 2   plan_type                   1000 non-null   object  
 3   monthly_fee                 1000 non-null   int64   
 4   acquisition_cost            1000 non-null   int64   
 5   churn_date                  168 non-null    object  
 6   is_active                   1000 non-null   int64   
 7   tenure_days                 1000 non-null   int64   
 8   tenure_month                1000 non-null   int64   
 9   tenure_cohort               1000 non-null   category
 10  total_subscription_revenue  1000 non-null   int64   
dtypes: category(1), int64(7), object(3)
memory usage: 79.4+ KB


In [143]:
customers.shape

(1000, 11)

In [144]:
customers.head()

,customer_id,signup_date,plan_type,monthly_fee,acquisition_cost,churn_date,is_active,tenure_days,tenure_month,tenure_cohort,total_subscription_revenue
0,1001,2024-11-07,Basic,50,30,NaT,1,235,8,6-9 Months,400
1,1002,2024-06-06,Basic,50,30,NaT,1,389,13,12-18 Months,650
2,1003,2024-12-31,Basic,50,30,NaT,1,181,7,6-9 Months,350
3,1004,2024-11-21,Pro,200,100,NaT,1,221,8,6-9 Months,1600
4,1005,2024-08-16,Pro,200,100,NaT,1,318,11,9-12 Months,2200


In [145]:
# Export the customer level dataset

customers.to_csv('dim_customers.csv',index=False)

In [146]:
# calculating total customer
total_customers = len(customers)

# Count customers who churned b
churned_customers = customers['churn_date'].notna().sum()

# Churn Rate
overall_churn_rate = (churned_customers / total_customers) * 100

In [147]:
# Calculate average tenure days
avg_tenure_days = customers['tenure_days'].mean()

In [148]:
# Calculate average tenure in months
avg_tenure_months = customers['tenure_month'].mean()

In [149]:
# plan-level kpi's
kpis = customers.groupby('plan_type').agg(
    total_users = ('customer_id','count'),
    churned_users = ('churn_date', lambda x: x.notna().sum()),
    avg_arpu = ('monthly_fee','mean'),
    avg_cac = ('acquisition_cost','mean'),
    avg_cumulative_subscription_revenue = ('total_subscription_revenue','mean')
).reset_index().round({'avg_cumulative_subscription_revenue': 2})    

In [150]:
#  Churn rate by plan
kpis['churn_rate_%'] = round((kpis['churned_users']/kpis['total_users']) * 100,2)

# Revenue-to-CAC ration,not a profit or margin calculation
kpis['revenue_cac_ratio'] = round(kpis['avg_cumulative_subscription_revenue']/kpis['avg_cac'],2)

In [151]:
kpis.head()

,plan_type,total_users,churned_users,avg_arpu,avg_cac,avg_cumulative_subscription_revenue,churn_rate_%,revenue_cac_ratio
0,Basic,327,58,50.0,30.0,445.72,17.74,14.86
1,Enterprise,330,57,500.0,200.0,4298.48,17.27,21.49
2,Pro,343,53,200.0,100.0,1773.18,15.45,17.73


In [152]:
# Revenue dataset columns names
revenue.columns

Index(['subscription_id', 'customer_id', 'month', 'monthly_fee',
       'revenue_type', 'amount'],
      dtype='object')

In [153]:
# MONTHLY LEDGER
churned_ledger = revenue[['customer_id', 'month', 'monthly_fee',
       'revenue_type']].copy()
churned_ledger['month'] = pd.to_datetime(churned_ledger['month']).dt.to_period('M').astype(str)
churned_ledger['customer_status'] = 'Churned'

# Generate missing customer month records for churned customers
rows = []
churned = customers[customers['churn_date'].notna()]

for i, customer in churned.iterrows():
    months = pd.period_range(
        customer['signup_date'],
        customer['churn_date'],
        freq = 'M'
    ).astype(str)
    for month in months:
        exists = (
            (churned_ledger['customer_id'] == customer['customer_id'])
            &
            (churned_ledger['month'] == month)
        ).any()

# Adding a record only when that customer month is missing
        if not exists:
            rows.append([
                customer['customer_id'],
                month,
                customer['monthly_fee'],
                'MRR',
                'Churned'
            ])
# Converting the records into a dataframe
missing_months  = pd.DataFrame(rows,
                               columns = [
                                   'customer_id',
                                   'month',
                                   'monthly_fee',
                                   'revenue_type',
                                   'customer_status'
                               ])

# Generate monthly records form signup through June 2025

rows = []
active = customers[customers['churn_date'].isna()]
             
for i, customer in active.iterrows():
    months = pd.period_range(
        start = customer['signup_date'], end = cutoff_date, freq = 'M'
    )
    for month in months :
        rows.append([
            customer['customer_id'],
            str(month),
            customer['monthly_fee'],
            'MRR',
            'Active'
        ])

# Converting records to dataframe
active_ledger = pd.DataFrame(rows,
                             columns = [
                                 'customer_id',
                                 'month',
                                 'monthly_fee',
                                 'revenue_type',
                                 'customer_status']
                            )
# Combine original churned billing records,missing churned months and active customers records
revenue_ledger = pd.concat([churned_ledger,missing_months,active_ledger],ignore_index = True)

# Sort records by customer and billing month
revenue_ledger = revenue_ledger.sort_values(['customer_id','month']).reset_index(drop=True)                      

In [160]:
revenue_ledger.head(20)

,customer_id,month,monthly_fee,revenue_type,customer_status
0,1001,2024-11,50,MRR,Active
1,1001,2024-12,50,MRR,Active
2,1001,2025-01,50,MRR,Active
3,1001,2025-02,50,MRR,Active
4,1001,2025-03,50,MRR,Active
5,1001,2025-04,50,MRR,Active
6,1001,2025-05,50,MRR,Active
7,1001,2025-06,50,MRR,Active
8,1002,2024-06,50,MRR,Active
9,1002,2024-07,50,MRR,Active


In [155]:
revenue_ledger.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8793 entries, 0 to 8792
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   customer_id      8793 non-null   int64 
 1   month            8793 non-null   object
 2   monthly_fee      8793 non-null   int64 
 3   revenue_type     8793 non-null   object
 4   customer_status  8793 non-null   object
dtypes: int64(2), object(3)
memory usage: 343.6+ KB


In [156]:
revenue_ledger.shape

(8793, 5)

In [157]:
# Export the monthly billing fact table

revenue_ledger.to_csv('fact_revenue.csv',index = False)